In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error

import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(1234567890)

SEQ_LENGTH = 520
HIDDEN_SIZE = 64
NUM_LAYERS = 2
LEARNING_RATE = 0.001
EPOCHS = 100
BATCH_SIZE = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
mat_contents = sio.loadmat('Xtrain.mat')
mat_test = sio.loadmat('Xtest.mat')

var_name = [key for key in mat_contents.keys() if not key.startswith('__')][0]
data = mat_contents[var_name].flatten()

var_name_test = [key for key in mat_test.keys() if not key.startswith('__')][0]
data_test = mat_test[var_name_test].flatten()

indices_test = np.arange(len(data), len(data) + len(data_test))

plt.figure(figsize=(12, 5))

plt.plot(data, label='Raw Laser Measurement (Train)', color='royalblue')

plt.plot(indices_test, data_test, label='Test Data', color='darkorange')

plt.title('Sequential Visualization of Training and Test Data')
plt.xlabel('Time Step')
plt.ylabel('Measurement')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(data.reshape(-1, 1)).flatten()

def create_sequences(data, seq_length):
    xs = []
    ys = []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

X, y = create_sequences(data_scaled, SEQ_LENGTH)

train_size = int(len(X) * 0.8)
X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:], y[train_size:]

X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1).to(device)
X_val_t = torch.tensor(X_val, dtype=torch.float32).unsqueeze(-1).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(-1).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, output_size=1):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        
        out, _ = self.lstm(x, (h0, c0))
        
        out = self.fc(out[:, -1, :])
        return out
    
class CNN_LSTM_Model(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
        super(CNN_LSTM_Model, self).__init__()
        
        self.conv1d = nn.Conv1d(in_channels=input_size, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        
        self.lstm = nn.LSTM(input_size=16, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = x.permute(0, 2, 1) 
        
        x = self.relu(self.conv1d(x))
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.lstm(x)
        
        out = self.fc(out[:, -1, :])
        return out

model = CNN_LSTM_Model(input_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, output_size=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train()
    batch_losses = []
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    train_losses.append(np.mean(batch_losses))

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_t)
        val_loss = criterion(val_outputs, y_val_t)
        val_losses.append(val_loss.item())

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_losses[-1]:.6f}, Val Loss: {val_losses[-1]:.6f}')

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    X_all_t = torch.tensor(X, dtype=torch.float32).unsqueeze(-1).to(device)
    predictions_scaled = model(X_all_t).cpu().numpy()

predictions = scaler.inverse_transform(predictions_scaled).flatten()
actual_y = scaler.inverse_transform(y.reshape(-1, 1)).flatten()

time_steps = np.arange(SEQ_LENGTH, len(data))

plt.figure(figsize=(14, 6))
plt.plot(time_steps, actual_y, label='Actual Data', color='royalblue', alpha=0.7)
plt.plot(time_steps, predictions, label='1-Step Ahead Prediction', color='darkorange', linestyle='--', alpha=0.9)
plt.title('Model Performance: 1-Step Ahead Predictions Overlay on Training Data')
plt.xlabel('Time Step')
plt.ylabel('Measurement')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
current_seq = X[-1].copy()
predictions_scaled = []
predictions_amount = 400

model.eval()
with torch.no_grad():
    for _ in range(predictions_amount):
        seq_t = torch.tensor(current_seq, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
        
        next_pred = model(seq_t).cpu().numpy()[0, 0]
        predictions_scaled.append(next_pred)
        
        current_seq = np.append(current_seq[1:], next_pred)

predictions = scaler.inverse_transform(np.array(predictions_scaled).reshape(-1, 1)).flatten()

forecast_steps = len(data_test)
future_time_steps = np.arange(len(data), len(data) + forecast_steps)

plt.figure(figsize=(14, 6))
plt.plot(np.arange(len(data)), data, label='Historical Data (Train)', color='royalblue')

plt.plot(future_time_steps, data_test, label='Actual Future Data (Test)', color='darkorange')
plt.plot(future_time_steps, predictions[:forecast_steps], label='Recursive Prediction', color='crimson')

plt.title('Recursive Prediction ' + str(predictions_amount) + ' Steps into the Future')
plt.xlabel('Time Step')
plt.ylabel('Measurement')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error

seq_length_values = range(200, min(600, len(data)), 20)

all_results = []

for seq_length in seq_length_values:
    print(f"\nEvaluating SEQ_LENGTH: {seq_length}")
    set_seed(1234567890)
    
    scaler_iter = MinMaxScaler(feature_range=(-1, 1))
    data_scaled_iter = scaler_iter.fit_transform(data.reshape(-1, 1)).flatten()

    def create_sequences_iter(data_array, sequence_length):
        inputs = []
        targets = []
        for index in range(len(data_array) - sequence_length):
            inputs.append(data_array[index:index + sequence_length])
            targets.append(data_array[index + sequence_length])
        return np.array(inputs), np.array(targets)

    X_iter, y_iter = create_sequences_iter(data_scaled_iter, seq_length)

    train_size_iter = int(len(X_iter) * 0.8)
    X_train_iter, y_train_iter = X_iter[:train_size_iter], y_iter[:train_size_iter]
    X_val_iter, y_val_iter = X_iter[train_size_iter:], y_iter[train_size_iter:]

    X_train_iter_t = torch.tensor(X_train_iter, dtype=torch.float32).unsqueeze(-1).to(device)
    y_train_iter_t = torch.tensor(y_train_iter, dtype=torch.float32).unsqueeze(-1).to(device)
    X_val_iter_t = torch.tensor(X_val_iter, dtype=torch.float32).unsqueeze(-1).to(device)
    y_val_iter_t = torch.tensor(y_val_iter, dtype=torch.float32).unsqueeze(-1).to(device)

    train_dataset_iter = TensorDataset(X_train_iter_t, y_train_iter_t)
    train_loader_iter = DataLoader(train_dataset_iter, batch_size=BATCH_SIZE, shuffle=True)

    model_iter = CNN_LSTM_Model(input_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, output_size=1).to(device)
    criterion_iter = nn.MSELoss()
    optimizer_iter = torch.optim.Adam(model_iter.parameters(), lr=LEARNING_RATE)

    # Train model
    for epoch in range(EPOCHS):
        model_iter.train()
        for batch_X, batch_y in train_loader_iter:
            optimizer_iter.zero_grad()
            outputs = model_iter(batch_X)
            loss = criterion_iter(outputs, batch_y)
            loss.backward()
            optimizer_iter.step()

    # Recursive Prediction on Test Set
    current_seq_iter = X_iter[-1].copy()
    predictions_scaled_iter = []
    forecast_steps = len(data_test)

    model_iter.eval()
    with torch.no_grad():
        for _ in range(forecast_steps):
            seq_t = torch.tensor(current_seq_iter, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
            next_pred = model_iter(seq_t).cpu().numpy()[0, 0]
            predictions_scaled_iter.append(next_pred)
            current_seq_iter = np.append(current_seq_iter[1:], next_pred)

    # Invert scaling
    predictions_iter = scaler_iter.inverse_transform(np.array(predictions_scaled_iter).reshape(-1, 1)).flatten()
    
    # Calculate MAE and MSE for this seq_length
    mae_val = mean_absolute_error(data_test, predictions_iter)
    mse_val = mean_squared_error(data_test, predictions_iter)
    
    all_results.append({
        'seq_length': seq_length,
        'mae': mae_val,
        'mse': mse_val,
        'predictions': predictions_iter,
        'historical_data': scaler_iter.inverse_transform(data_scaled_iter.reshape(-1, 1)).flatten()
    })
    print(f"Finished SEQ_LENGTH {seq_length}: MAE={mae_val:.4f}, MSE={mse_val:.4f}")

# Sort results by MSE to find best 3
best_results = sorted(all_results, key=lambda x: x['mse'])[:3]

print("\n--- Top 3 Best Performing Models (by MSE) ---")

# Display results for top 3 (not used for report but just to see how close we can get with brute-force search)
for i, res in enumerate(best_results):
    seq_l = res['seq_length']
    preds = res['predictions']
    hist = res['historical_data']
    
    num_test_points = len(data_test)
    test_predictions = preds[:num_test_points]
    actual_test_values = data_test

    mae = mean_absolute_error(actual_test_values, test_predictions)
    mse = mean_squared_error(actual_test_values, test_predictions)

    print(f"\nRank {i+1}: SEQ_LENGTH={seq_l}")
    print(f"Model Evaluation on Test Dataset")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"Mean Squared Error (MSE): {mse:.4f}")

    # Plot 1: Full Recursive Prediction (History + Forecast)
    plt.figure(figsize=(14, 6))
    plt.plot(np.arange(len(hist)), hist, label='Historical Data', color='royalblue')
    plt.plot(np.arange(len(hist), len(hist) + len(preds)), preds, label=f'Recursive Prediction (SEQ={seq_l})', color='crimson')
    plt.plot(np.arange(len(hist), len(hist) + len(data_test)), data_test, label='Actual Test Data', color='darkorange', alpha=0.6)
    plt.axvline(len(hist) - 1, color='gray', linestyle='--')
    plt.title(f'Rank {i+1} Best Performing: Recursive Prediction with SEQ_LENGTH={seq_l}')
    plt.xlabel('Time Step')
    plt.ylabel('Measurement')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Plot 2: Visual Assessment (Zoomed into Test Set only)
    plt.figure(figsize=(12, 6))
    plt.plot(actual_test_values, label='Actual Test Values', color='blue', linewidth=2)
    plt.plot(test_predictions, label='Model Predictions', color='crimson', linestyle='--', linewidth=2)
    plt.title(f'Rank {i+1} Visual Assessment: Predicted vs Real Values (SEQ_LENGTH={seq_l})')
    plt.xlabel('Time Step')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True)
    plt.show()

# Final Graph of MAE and MSE for the top 3
top_labels = [f"SEQ={r['seq_length']}" for r in best_results]
top_maes = [r['mae'] for r in best_results]
top_mses = [r['mse'] for r in best_results]

fig, ax1 = plt.subplots(figsize=(10, 6))

color = 'tab:blue'
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('MAE', color=color)
ax1.bar(top_labels, top_maes, color=color, alpha=0.6, label='MAE')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('MSE', color=color)
ax2.plot(top_labels, top_mses, color=color, marker='o', linewidth=2, label='MSE')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('MAE and MSE Comparison for Top 3 Sequence Lengths')
fig.tight_layout()
plt.show()


In [ ]:
num_test_points = len(data_test)
test_predictions = predictions[:num_test_points]
actual_test_values = data_test

mae = mean_absolute_error(actual_test_values, test_predictions)
mse = mean_squared_error(actual_test_values, test_predictions)

print(f"Model Evaluation on Test Dataset")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")

plt.figure(figsize=(12, 6))
plt.plot(actual_test_values, label='Actual Test Values', color='blue', linewidth=2)
plt.plot(test_predictions, label='Model Predictions', color='crimson', linestyle='--', linewidth=2)
plt.title('Visual Assessment: Predicted vs Real Values (Test Set)')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()